# Sentiment Analysis with TF–IDF and Logistic Regression

This notebook trains a simple, interpretable text classifier to predict sentiment labels from the `clean_text` column. It checks the data, prepares a stratified train/test split, converts text to TF–IDF features, and evaluates the model on held-out examples.

**Labels:** `-1` = negative, `0` = neutral, `1` = positive. The test set is kept separate from vectorizer fitting and model training so the reported scores measure performance on unseen rows.


## 1. Imports and settings

Set the random seed and test fraction here so the split is reproducible and easy to adjust.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 23
TEST_SIZE = 0.20
DATA_PATH = Path("data/sentiment.csv")


## 2. Load and inspect the data

The model needs a text field and a sentiment label. This cell reports the dataset dimensions, missing values, and a few examples before any rows are removed.

In [ ]:
df = pd.read_csv(DATA_PATH)
required_columns = {"clean_text", "category"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
display(df.head())
display(df[["clean_text", "category"]].isna().sum().rename("missing rows"))


## 3. Check label balance and prepare rows

Rows without text or a label cannot be used for supervised training, so they are removed. The chart helps show whether the classes are balanced; the split below preserves their proportions.

In [ ]:
print("Label counts before dropping incomplete rows:")
display(df["category"].value_counts(dropna=False).sort_index().rename("rows"))

rows_before = len(df)
df = df.dropna(subset=["clean_text", "category"]).copy()
df["clean_text"] = df["clean_text"].astype(str)
if df.empty:
    raise ValueError("No rows with both text and a category remain.")

print(f"Dropped {rows_before - len(df):,} incomplete rows; {len(df):,} rows remain.")
display(df["category"].value_counts().sort_index().rename("rows"))
ax = df["category"].value_counts().sort_index().plot.bar(color="#4472C4", rot=0)
ax.set(title="Sentiment label distribution", xlabel="Sentiment label", ylabel="Rows")
plt.tight_layout()
plt.show()


## 4. Normalize text and create a stratified split

Text is lowercased and URLs, user mentions, and non-letter characters are removed. The vectorizer is fitted on training text only, which avoids leaking information from the test set into feature construction.

In [ ]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

X = df["clean_text"].map(normalize_text)
y = df["category"]
if y.nunique() < 2:
    raise ValueError("At least two distinct sentiment labels are required.")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

vectorizer = TfidfVectorizer(max_features=7_000, min_df=2)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Training rows: {len(X_train):,} | Test rows: {len(X_test):,}")
print(f"Vocabulary size: {len(vectorizer.get_feature_names_out()):,}")


## 5. Train the classifier

Logistic regression is a strong baseline for sparse TF–IDF text features. `max_iter` gives the optimizer room to converge.

In [ ]:
model = LogisticRegression(max_iter=1_000, random_state=RANDOM_STATE)
model.fit(X_train_tfidf, y_train)
print("Model training complete.")


## 6. Evaluate on the held-out test set

Accuracy is the share of correct predictions. The per-class precision, recall, and F1 scores give a more complete view, especially if classes have different frequencies.

In [ ]:
y_pred = model.predict(X_test_tfidf)
labels = model.classes_

print(f"Test accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(
    classification_report(
        y_test,
        y_pred,
        labels=labels,
        zero_division=0,
        target_names=[str(label) for label in labels],
    )
)


## 7. Confusion matrix

Rows show the true label and columns show the predicted label. The diagonal contains correct predictions; off-diagonal counts show which sentiments the model confuses.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=labels,
    display_labels=[str(label) for label in labels],
    cmap="Blues",
    values_format="d",
    ax=ax,
)
ax.set_title("Sentiment classification on the test set")
plt.tight_layout()
plt.show()


## Takeaway

Use the per-class report and confusion matrix together to identify where the baseline performs well and where it needs improvement. Scores depend on this dataset and split; they do not guarantee performance on new sources or changing language. Possible next steps include reviewing misclassified examples, comparing preprocessing choices, and tuning regularization with cross-validation.